# CRM Pipeline: Pull → Dedup → Enrich → Report

End-to-end CRM integration demo using `siege_utilities.connectors`.

Sections:
1. Setup and credentials
2. Pull contacts from multiple CRMs
3. Cross-CRM deduplication
4. Geographic enrichment
5. Sales pipeline report
6. Write-back canonical IDs

## 1. Setup

All credentials from environment variables — never hardcoded.

In [ ]:
import os
import logging

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(name)s %(message)s")
log = logging.getLogger(__name__)

## 2. Pull Contacts from Multiple CRMs

Each connector follows the same protocol — authenticate, then fetch.

In [ ]:
from siege_utilities.connectors import SalesforceConnector, HubSpotConnector

# --- Salesforce ---
sf = SalesforceConnector(
    client_id=os.environ.get("SF_CLIENT_ID", ""),
    client_secret=os.environ.get("SF_CLIENT_SECRET", ""),
    username=os.environ.get("SF_USERNAME"),
    password=os.environ.get("SF_PASSWORD"),
    security_token=os.environ.get("SF_SECURITY_TOKEN"),
)
sf.authenticate()

sf_contacts = sf.get_objects("Contact", limit=500)
sf_opps = sf.get_objects("Opportunity", limit=500)
print(f"Salesforce: {len(sf_contacts)} contacts, {len(sf_opps)} opportunities")

In [ ]:
# --- HubSpot ---
hs = HubSpotConnector(access_token=os.environ.get("HUBSPOT_TOKEN", ""))
hs.authenticate()

hs_contacts = hs.get_objects("contacts", limit=500)
hs_deals = hs.get_objects("deals", limit=500)
print(f"HubSpot: {len(hs_contacts)} contacts, {len(hs_deals)} deals")

## 3. Cross-CRM Deduplication

Convert to canonical CRM models, then run the dedup pipeline.

In [ ]:
from siege_utilities.connectors import CRMContact, crm_dedup_pipeline

# Map to canonical models
sf_crm = sf.to_crm_contacts(sf_contacts)
hs_crm = hs.to_crm_contacts(hs_contacts)

# Convert to DataFrames
sf_df = CRMContact.to_dataframe(sf_crm)
hs_df = CRMContact.to_dataframe(hs_crm)

print(f"Salesforce canonical: {len(sf_df)} contacts")
print(f"HubSpot canonical: {len(hs_df)} contacts")

In [ ]:
# Run dedup pipeline
merge_table = crm_dedup_pipeline(
    [sf_df, hs_df],
    name_columns=["first_name", "last_name"],
)

duplicates = merge_table[merge_table.duplicated("canonical_id", keep=False)]
print(f"Merge table: {len(merge_table)} records")
print(f"Cross-system matches: {len(duplicates)} records")
merge_table.head(10)

## 4. Geographic Enrichment

Extract addresses for geocoding via `geo/` providers.

In [ ]:
from siege_utilities.connectors import geographic_adapter

# Prepare addresses for geocoding
geo_ready = geographic_adapter(sf_df)
print(f"Records with addresses: {len(geo_ready)}")
geo_ready.head()

## 5. Sales Pipeline Report

Build a pipeline chart from opportunity data.

In [ ]:
from siege_utilities.connectors import pipeline_adapter, tabular_adapter

# Salesforce opportunities → pipeline shape
pipeline_data = pipeline_adapter(
    sf_opps,
    stage_column="StageName",
    value_column="Amount",
)
print("Pipeline summary:")
pipeline_data

In [ ]:
# Top opportunities table
top_opps = tabular_adapter(
    sf_opps,
    columns=["Name", "StageName", "Amount", "CloseDate", "Probability"],
    rename={"StageName": "Stage", "CloseDate": "Close Date"},
    sort_by="Amount",
    ascending=False,
    limit=20,
)
top_opps

## 6. Write-Back Canonical IDs

Enrich source records with canonical IDs from the dedup pipeline.

In [ ]:
# Merge canonical IDs back to source DataFrames
sf_enriched = sf_df.merge(
    merge_table[["source_id", "canonical_id"]],
    left_on="source_id",
    right_on="source_id",
    how="left",
)
print(f"Enriched Salesforce contacts: {len(sf_enriched)} records")
print(f"With canonical ID: {sf_enriched['canonical_id'].notna().sum()}")
sf_enriched[["first_name", "last_name", "email", "canonical_id"]].head(10)

In [ ]:
# Clean up
sf.close()
hs.close()
print("Done.")